# K. Pruebas automatizadas, API, MLflow y Docker

Este notebook documenta la validación del proyecto **Adult Income**. Las celdas están preparadas para ejecutarse manualmente y cubren dependencias, pruebas de datos, modelo y API, consulta del manifiesto, validación de endpoints y comprobación del contenedor Docker.

## 1. Verificación de la estructura del proyecto

In [4]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
print(f'Proyecto: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')
print(f'Versión: {sys.version}')

required_files = [
    'requirements-dev.txt',
    'Dockerfile',
    'src/api/main.py',
    'tests/__init__.py',
    'tests/test_data.py',
    'tests/test_model.py',
    'tests/test_api.py',
    'resultado_pipeline/adult_clean.csv',
    'resultado_pipeline/modelo/modelo_clasificacion.joblib',
    'resultado_pipeline/modelo/manifest_modelo.json',
]

for relative_path in required_files:
    status = 'OK' if (PROJECT_ROOT / relative_path).exists() else 'FALTA'
    print(f'[{status}] {relative_path}')

Proyecto: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
Python: c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
Versión: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
[OK] requirements-dev.txt
[OK] Dockerfile
[OK] src/api/main.py
[OK] tests/__init__.py
[OK] tests/test_data.py
[OK] tests/test_model.py
[OK] tests/test_api.py
[OK] resultado_pipeline/adult_clean.csv
[OK] resultado_pipeline/modelo/modelo_clasificacion.joblib
[OK] resultado_pipeline/modelo/manifest_modelo.json


## 2. Instalación y verificación de dependencias

La siguiente celda instala todo utilizando exactamente el mismo Python que ejecuta este notebook. No depende de activar un entorno desde una terminal. En Python 3.13 instala versiones recientes de FastAPI y Pydantic con soporte para ese intérprete, evitando el error de compilación de `pydantic-core==2.18.2`.

In [1]:
import subprocess
import sys

print(f'Instalando con: {sys.executable}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    '-r', 'requirements-dev.txt'
])
print('Dependencias instaladas. Reinicie el kernel y continúe con la celda siguiente.')

Instalando con: c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe


Dependencias instaladas. Reinicie el kernel y continúe con la celda siguiente.


### Reinicio manual del kernel

Después de completar la instalación, reinicie manualmente el kernel desde **Restart Kernel** o **Kernel > Restart**. VS Code puede reportar falsamente un bloqueo cuando el reinicio se solicita mediante código. Después del reinicio, continúe desde la celda de verificación de dependencias y no vuelva a ejecutar la instalación.

In [2]:
print('No ejecute código para apagar el kernel.')
print('Use el botón Restart Kernel y continúe con la siguiente sección.')

No ejecute código para apagar el kernel.
Use el botón Restart Kernel y continúe con la siguiente sección.


In [3]:
import fastapi
import httpx
import joblib
import pandas
import pytest
import sklearn

print(f'FastAPI: {fastapi.__version__}')
print(f'HTTPX: {httpx.__version__}')
print(f'Joblib: {joblib.__version__}')
print(f'Pandas: {pandas.__version__}')
print(f'Pytest: {pytest.__version__}')
print(f'Scikit-learn: {sklearn.__version__}')

FastAPI: 0.139.0
HTTPX: 0.27.2
Joblib: 1.5.3
Pandas: 2.3.3
Pytest: 8.3.3
Scikit-learn: 1.9.0


## 3. Pruebas automatizadas

Las pruebas se pueden ejecutar por separado para identificar con claridad cualquier fallo.

In [4]:
!pytest tests/test_data.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 17 items

tests/test_data.py::test_schema_has_exact_columns PASSED                 [  5%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[age] PASSED  [ 11%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[fnlwgt] PASSED [ 17%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[education-num] PASSED [ 23%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[capital-gain] PASSED [ 29%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[capital-loss] PASSED [ 35%]
tests/test_data.py::test_numeric_columns_have_numeric_dtype[hours-per-week] PASSED [ 41%]
tests/test_data.py::test_numeric_columns_have_nume

In [5]:
!pytest tests/test_model.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 11 items

tests/test_model.py::test_model_file_exists PASSED                       [  9%]
tests/test_model.py::test_model_loads_without_error PASSED               [ 18%]
tests/test_model.py::test_model_exposes_predict PASSED                   [ 27%]
tests/test_model.py::test_model_exposes_predict_proba PASSED             [ 36%]
tests/test_model.py::test_valid_input_returns_one_prediction PASSED      [ 45%]
tests/test_model.py::test_valid_input_returns_binary_prediction PASSED   [ 54%]
tests/test_model.py::test_model_returns_two_probabilities PASSED         [ 63%]
tests/test_model.py::test_probabilities_are_between_zero_and_one PASSED  [ 72%]
tests/t

In [6]:
!pytest tests/test_api.py -v --disable-warnings

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 37 items

tests/test_api.py::test_health_returns_200 PASSED                        [  2%]
tests/test_api.py::test_health_returns_model_metadata PASSED             [  5%]
tests/test_api.py::test_valid_request_returns_200 PASSED                 [  8%]
tests/test_api.py::test_valid_request_returns_expected_schema PASSED     [ 10%]
tests/test_api.py::test_prediction_is_binary PASSED                      [ 13%]
tests/test_api.py::test_probability_is_valid PASSED                      [ 16%]
tests/test_api.py::test_model_version_is_valid PASSED                    [ 18%]
tests/test_api.py::test_out_of_range_values_return_422[age-16] PASSED    [ 21%]
tests/t

### Ejecución completa

Esta celda ejecuta toda la suite y muestra el resumen general.

In [7]:
!pytest tests/ -v --disable-warnings

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 65 items

tests/test_api.py::test_health_returns_200 PASSED                        [  1%]
tests/test_api.py::test_health_returns_model_metadata PASSED             [  3%]
tests/test_api.py::test_valid_request_returns_200 PASSED                 [  4%]
tests/test_api.py::test_valid_request_returns_expected_schema PASSED     [  6%]
tests/test_api.py::test_prediction_is_binary PASSED                      [  7%]
tests/test_api.py::test_probability_is_valid PASSED                      [  9%]
tests/test_api.py::test_model_version_is_valid PASSED                    [ 10%]
tests/test_api.py::test_out_of_range_values_return_422[age-16] PASSED    [ 12%]
tests/t

### Guardar evidencia de pytest

La celda siguiente vuelve a ejecutar la suite y guarda la salida en `resultado_pruebas.txt`. Un código de salida igual a `0` indica que todas las pruebas ejecutadas pasaron.

In [9]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests', '-v', '--disable-warnings'],
    capture_output=True,
    text=True,
)
report = result.stdout
if result.stderr:
    report += '\n\nERRORES:\n' + result.stderr
report_path = PROJECT_ROOT / 'resultado_pruebas.txt'
report_path.write_text(report, encoding='utf-8')
print(report)
print(f'Código de salida: {result.returncode}')
print(f'Reporte: {report_path}')

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 65 items

tests/test_api.py::test_health_returns_200 PASSED                        [  1%]
tests/test_api.py::test_health_returns_model_metadata PASSED             [  3%]
tests/test_api.py::test_valid_request_returns_200 PASSED                 [  4%]
tests/test_api.py::test_valid_request_returns_expected_schema PASSED     [  6%]
tests/test_api.py::test_prediction_is_binary PASSED                      [  7%]
tests/test_api.py::test_probability_is_valid PASSED                      [  9%]
tests/test_api.py::test_model_version_is_valid PASSED                    [ 10%]
tests/test_api.py::test_out_of_range_values_return_422[age-16] PASSED    [ 12%]
tests/t

## 4. Resumen del modelo seleccionado

In [10]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
manifest_path = PROJECT_ROOT / 'resultado_pipeline/modelo/manifest_modelo.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
summary = {
    'algoritmo_seleccionado': manifest.get('selected_algorithm'),
    'metrica_seleccion': manifest.get('selection_metric'),
    'umbral': manifest.get('selected_threshold'),
    'metricas_test': manifest.get('test_metrics'),
    'runs_mlflow': manifest.get('mlflow_run_ids'),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "algoritmo_seleccionado": "hist_gradient_boosting",
  "metrica_seleccion": "average_precision",
  "umbral": 0.6449999999999999,
  "metricas_test": {
    "threshold": 0.6449999999999999,
    "precision": 0.6847988077496274,
    "recall": 0.7868150684931506,
    "f1": 0.7322709163346613,
    "roc_auc": 0.9331584725436779,
    "average_precision": 0.8406640269639136,
    "balanced_accuracy": 0.8364148099135115
  },
  "runs_mlflow": {
    "hist_gradient_boosting": "43c049b122b0444ab1e259683f73cdc2",
    "random_forest": "a7cd9358709f47e8863200479157f860",
    "logistic_regression": "be4b4b8b80834fc8ae7d806a9d986f48",
    "dummy_baseline": "dd3d21e492ed4cd7b4aceb2ee45033c1",
    "selected_model": "225a40e0708743a4a1ec292458935b43"
  }
}


## 5. API local

Antes de ejecutar las siguientes celdas, inicie la API en una terminal separada:

```powershell
python -m uvicorn src.api.main:app --reload --host 127.0.0.1 --port 8001
```

Swagger estará disponible en `http://127.0.0.1:8001/docs`.

In [11]:
import json
import httpx

response = httpx.get('http://127.0.0.1:8001/health', timeout=10)
print(f'HTTP {response.status_code}')
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

HTTP 200
{
  "status": "ok",
  "algorithm": "hist_gradient_boosting",
  "threshold": 0.645,
  "model_version": "225a40e0"
}


In [12]:
import json
import httpx

payload = {
    'age': 39,
    'workclass': 'State-gov',
    'fnlwgt': 77516,
    'education': 'Bachelors',
    'education-num': 13,
    'marital-status': 'Never-married',
    'occupation': 'Adm-clerical',
    'relationship': 'Not-in-family',
    'race': 'White',
    'sex': 'Male',
    'capital-gain': 2174,
    'capital-loss': 0,
    'hours-per-week': 40,
    'native-country': 'United-States',
}
response = httpx.post(
    'http://127.0.0.1:8001/predict', json=payload, timeout=30
)
print(f'HTTP {response.status_code}')
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

HTTP 200
{
  "prediction": 0,
  "probability": 0.0131,
  "model_version": "225a40e0"
}


### Validación de una entrada incorrecta

Una edad inferior a 17 debe ser rechazada con HTTP 422.

In [13]:
import json
import httpx

payload = {
    'age': 39,
    'workclass': 'State-gov',
    'fnlwgt': 77516,
    'education': 'Bachelors',
    'education-num': 13,
    'marital-status': 'Never-married',
    'occupation': 'Adm-clerical',
    'relationship': 'Not-in-family',
    'race': 'White',
    'sex': 'Male',
    'capital-gain': 2174,
    'capital-loss': 0,
    'hours-per-week': 40,
    'native-country': 'United-States',
}
invalid_payload = payload.copy()
invalid_payload['age'] = 10
response = httpx.post(
    'http://127.0.0.1:8001/predict', json=invalid_payload, timeout=30
)
print(f'HTTP {response.status_code}')
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

HTTP 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "age"
      ],
      "msg": "Input should be greater than or equal to 17",
      "input": 10,
      "ctx": {
        "ge": 17
      }
    }
  ]
}


## 6. MLflow

MLflow no es necesario para ejecutar pytest ni para que la API cargue el `.joblib`. Para revisar los experimentos, actívelo en una terminal separada:

```powershell
mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
```

Después abra `http://127.0.0.1:5000`.

## 7. Docker

Construya y ejecute la imagen desde una terminal separada:

```powershell
docker build -t grupo2-mlops .
docker run --rm -p 8001:8000 grupo2-mlops
```

El puerto `8001` corresponde a la computadora y el puerto `8000` al contenedor.

In [16]:
import json
import httpx

response = httpx.get('http://localhost:8001/health', timeout=10)
print(f'HTTP {response.status_code}')
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

HTTP 200
{
  "status": "ok",
  "algorithm": "hist_gradient_boosting",
  "threshold": 0.645,
  "model_version": "225a40e0"
}


In [17]:
import json
import httpx

payload = {
    'age': 39,
    'workclass': 'State-gov',
    'fnlwgt': 77516,
    'education': 'Bachelors',
    'education-num': 13,
    'marital-status': 'Never-married',
    'occupation': 'Adm-clerical',
    'relationship': 'Not-in-family',
    'race': 'White',
    'sex': 'Male',
    'capital-gain': 2174,
    'capital-loss': 0,
    'hours-per-week': 40,
    'native-country': 'United-States',
}
response = httpx.post(
    'http://localhost:8001/predict', json=payload, timeout=30
)
print(f'HTTP {response.status_code}')
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

HTTP 200
{
  "prediction": 0,
  "probability": 0.0131,
  "model_version": "225a40e0"
}


## 8. Conclusión

Al completar las celdas se obtiene evidencia de que los datos cumplen el esquema, el pipeline puede cargarse, las probabilidades son válidas, la API acepta y rechaza entradas correctamente, y el servicio funciona tanto localmente como dentro de Docker.